# 6a단계: 추적 · 팀 라벨 안정화 (선수 트레이싱을 위한 준비)

선수 한 명을 경기 내내 따라가려면 **같은 사람은 계속 같은 ID**, **ID마다 맞는 팀**이 필요합니다.
v6은 엉킨 무리에서 추적 ID가 다른 팀 선수로 넘어가면서 홍익↔국민대 오분류가 36건 생겼고,
긴 클립에서는 ROI 갱신이 한 번 잘못돼 인원이 튀었습니다. 이 노트북은 이 두 가지를 고치고, 득점 선수 정답 경로도 확인합니다.

**실행 순서:** 공통 셀 → ①(GPU) → (런타임 끊고 CPU로 공통 셀 다시) → ② → ③ → ④

## 공통 설정

In [ ]:
!pip install -q ultralytics
import os, json, glob, shutil, zipfile, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

# 한글 글꼴 (그래프의 한글이 네모로 깨지지 않게)
import subprocess, matplotlib.font_manager as fm
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(_font):
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
else:
    print("⚠️ 한글 글꼴 설치 실패: 그래프의 한글이 네모로 보일 수 있어요")
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# ===== 경로 (v3 노트북 Cell 2와 동일하게) =====
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")   # v3의 tracking_final.json 위치
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")           # 이번에 새로 만드는 평가 폴더
IMG_DIR          = os.path.join(EVAL_DIR, "images")                 # 평가용 프레임 20장 (정답 라벨의 기준 이미지)
PRE_DIR          = os.path.join(EVAL_DIR, "prelabels")              # 사전 라벨
for _d in (EVAL_DIR, IMG_DIR, PRE_DIR):
    os.makedirs(_d, exist_ok=True)

# ===== 클래스 =====
# 정답(GT)에는 0~2만 남아야 합니다. 3(Unknown)은 "라벨러가 확인해야 할 박스" 표시용입니다.
CLASS_NAMES = ["Hongik", "Kookmin", "Referee", "Unknown"]
CLS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# ===== 평가 클립 =====
# play_range : 득점 플레이 구간 (v3 frame_id 기준, 1부터 시작, 양 끝 포함)
# scorer     : v3에서 득점 선수 경로를 만들 때 쓴 (track_id, 프레임 구간). 득점 선수 '정답 위치'로 사용.
#              ⚠️ 0156의 #171, #183 구간은 v3 노트북 주석에 '추정, 확인 필요'로 되어 있었습니다.
#                 v3의 프레임 브라우저 셀로 확인 후 틀렸다면 여기 숫자를 고치세요.
EVAL_CLIPS = {
    "DJI_20250913133612_0144_D": {
        "clip_key": 34,
        "play_range": (413, 649),
        "scorer": [(2122, (413, 649))],
    },
    "DJI_20250913135020_0156_D": {
        "clip_key": 44,
        "play_range": (167, 525),
        "scorer": [(184, (167, 214)), (171, (215, 313)), (183, (314, 393)), (208, (410, 525))],
    },
}
FRAMES_PER_CLIP = 10

# ===== 매칭 기준 =====
# 선수가 10~25px로 작아서 IoU 0.5는 박스가 조금만 어긋나도 오답이 됩니다.
# 대신 '박스 중심 거리'로 짝을 짓습니다: 거리 <= max(MIN_PX, RATIO × 정답 박스 긴 변)
MATCH_MIN_PX = 10
MATCH_RATIO  = 0.5
SCORER_MATCH_PX = 15   # 득점 선수 정답 위치(발)와 예측 박스 발 위치가 이 거리(px) 이내면 '찾았다'


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")

def tracking_json_of(stem, folder=DRIVE_OUTPUT_DIR):
    return os.path.join(folder, f"{stem}_tracking_final.json")

for stem in EVAL_CLIPS:
    print(stem, "| 영상:", os.path.exists(video_path_of(stem)), "| v3 JSON:", os.path.exists(tracking_json_of(stem)))

In [ ]:
# ===== 공통 함수 =====

def load_tracking(json_path):
    """v3 형식 JSON → {frame_id: [player dict, ...]}, field_polygon"""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    by_frame = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    poly = np.array(data.get("field_polygon", []), dtype=np.int32)
    return by_frame, poly


def read_frames_sequential(video_path, frame_ids):
    """
    frame_id(1부터 시작)에 해당하는 프레임을 '처음부터 순서대로 읽어서' 가져옵니다.
    cap.set(POS_FRAMES)로 바로 점프하면 드론 영상(H.264/H.265)에서 몇 프레임씩 어긋날 수 있어서,
    v3가 프레임을 센 방식(처음부터 1, 2, 3...)과 똑같이 맞추기 위해 순차로 읽습니다.
    """
    wanted = set(frame_ids)
    last = max(wanted)
    out = {}
    cap = cv2.VideoCapture(video_path)
    fid = 0
    while fid < last:
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    missing = wanted - set(out)
    if missing:
        print(f"⚠️ 읽지 못한 프레임: {sorted(missing)}")
    return out


def pick_frames(play_range, n):
    a, b = play_range
    return sorted(set(np.linspace(a, b, n).round().astype(int).tolist()))


def inside(poly, x, y):
    if poly is None or len(poly) < 3:
        return True
    return cv2.pointPolygonTest(poly.reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def xyxy_to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    w, h = (x2 - x1) / W, (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def yolo_line_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()[:5]
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def box_iou(a, b):
    """Day 2에서 만든 IoU (xyxy)"""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_by_center(gt_boxes, pred_boxes):
    """
    정답 박스와 예측 박스를 1:1로 짝짓기 (헝가리안 알고리즘, 중심 거리 기준).
    return: [(gi, pi), ...], 짝 없는 정답 인덱스, 짝 없는 예측 인덱스
    """
    G, P = len(gt_boxes), len(pred_boxes)
    if G == 0 or P == 0:
        return [], list(range(G)), list(range(P))
    gc = np.array([center(b) for b in gt_boxes])
    pc = np.array([center(b) for b in pred_boxes])
    dist = np.linalg.norm(gc[:, None, :] - pc[None, :, :], axis=2)          # (G, P)
    gsize = np.array([max(b[2] - b[0], b[3] - b[1]) for b in gt_boxes])
    thr = np.maximum(MATCH_MIN_PX, MATCH_RATIO * gsize)[:, None]            # 정답 박스별 허용 거리
    cost = np.where(dist <= thr, dist, 1e6)
    gi, pi = linear_sum_assignment(cost)
    pairs = [(g, p) for g, p in zip(gi, pi) if cost[g, p] < 1e6]
    mg = {g for g, _ in pairs}
    mp = {p for _, p in pairs}
    return pairs, [g for g in range(G) if g not in mg], [p for p in range(P) if p not in mp]


print("✅ 공통 함수 준비 완료")

## 평가 준비 (0단계 파트 B와 같은 코드)

In [ ]:
GT_ZIP = os.path.join(EVAL_DIR, "gt_roboflow.zip")
GT_DIR = os.path.join("/content", "gt_extracted")
shutil.rmtree(GT_DIR, ignore_errors=True)
with zipfile.ZipFile(GT_ZIP) as z:
    z.extractall(GT_DIR)


def original_name(fname):
    """Roboflow가 바꾼 파일명 → 원래 이름. 예: 'DJI_..._f00413_jpg.rf.ab12cd.txt' → 'DJI_..._f00413'"""
    base = os.path.basename(fname)
    base = re.split(r"_(?:jpg|jpeg|png)\.rf\.", base, flags=re.IGNORECASE)[0]
    return os.path.splitext(base)[0]


def read_names_yaml(gt_dir):
    """Roboflow data.yaml의 클래스 순서를 읽음 (Roboflow가 순서를 바꿀 수 있어서 이름으로 다시 매핑)"""
    ymls = glob.glob(os.path.join(gt_dir, "**", "data.yaml"), recursive=True)
    if not ymls:
        return CLASS_NAMES
    txt = open(ymls[0], encoding="utf-8").read()
    m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
    if m:
        return [s.strip().strip("'\"") for s in m.group(1).split(",") if s.strip()]
    return re.findall(r"^\s*-\s*['\"]?([^'\"\n]+?)['\"]?\s*$", txt.split("names:")[1], re.M)


def load_gt(gt_dir):
    """→ {name: {"W","H","boxes":[(class_name, xyxy), ...]}}"""
    rf_names = read_names_yaml(gt_dir)
    gt = {}
    for lbl in glob.glob(os.path.join(gt_dir, "**", "labels", "*.txt"), recursive=True):
        name = original_name(lbl)
        img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))   # 원본 해상도 기준으로 복원
        if img is None:
            print("⚠️ 원본 이미지를 찾을 수 없음:", name); continue
        H, W = img.shape[:2]
        boxes = []
        for line in open(lbl).read().strip().splitlines():
            if not line.strip():
                continue
            c, b = yolo_line_to_xyxy(line, W, H)
            boxes.append((rf_names[c], b))
        gt[name] = {"W": W, "H": H, "boxes": boxes}
    return gt


GT = load_gt(GT_DIR)
_cnt = Counter(c for v in GT.values() for c, _ in v["boxes"])
print(f"정답 이미지 {len(GT)}장, 박스 {sum(_cnt.values())}개 → {dict(_cnt)}")
if _cnt.get("Unknown"):
    print(f"⚠️ 정답에 Unknown이 {_cnt['Unknown']}개 남아 있습니다. 탐지 평가에는 포함되지만 팀 정확도에서는 제외됩니다.")

In [ ]:
EVAL_TEAM = ["Hongik", "Kookmin", "Referee"]


def detection_and_team_metrics(run_json_by_stem):
    """정답 vs 예측: 탐지(클래스 무관) P/R/F1 + 짝지어진 박스의 클래스 정확도 + 혼동행렬"""
    tp = fp = fn = 0
    conf_mat = Counter()                     # (정답 클래스, 예측 클래스)
    per_frame = []
    for stem, jpath in run_json_by_stem.items():
        by_frame, _ = load_tracking(jpath)
        for name, g in GT.items():
            if not name.startswith(stem):
                continue
            fid = int(name.split("_f")[-1])
            preds = by_frame.get(fid, [])
            gb = [b for _, b in g["boxes"]]
            pb = [p["bbox"] for p in preds]
            pairs, ug, up = match_by_center(gb, pb)
            tp += len(pairs); fn += len(ug); fp += len(up)
            for gi, pi in pairs:
                conf_mat[(g["boxes"][gi][0], preds[pi]["type"])] += 1
            per_frame.append({"image": name, "gt": len(gb), "pred": len(pb), "TP": len(pairs), "FN": len(ug), "FP": len(up)})
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    team_pairs = {k: v for k, v in conf_mat.items() if k[0] in EVAL_TEAM}
    correct = sum(v for (g, p), v in team_pairs.items() if g == p)
    team_acc = correct / sum(team_pairs.values()) if team_pairs else 0.0
    ref_gt = sum(1 for v in GT.values() for c, _ in v["boxes"] if c == "Referee")
    ref_hit = conf_mat.get(("Referee", "Referee"), 0)
    return {
        "det_precision": round(P, 3), "det_recall": round(R, 3), "det_f1": round(F1, 3),
        "TP": tp, "FP": fp, "FN": fn,
        "team_acc": round(team_acc, 3),
        "referee_recall": round(ref_hit / ref_gt, 3) if ref_gt else None,
    }, conf_mat, pd.DataFrame(per_frame)


def scorer_track_metrics(stem, run_json, cfg, verbose=True):
    """
    득점 선수 추적 품질 (라벨링 없이 측정).
    - 정답 위치: v3에서 사람이 확인해 이어붙인 득점 선수의 발 위치 (cfg["scorer"])
    - 평가할 결과(run_json)에서 매 프레임 그 위치에 가장 가까운 박스의 track_id를 읽음

    지표
    - scorer_ids / scorer_id_switches : '정답 위치가 확실한 프레임'에서만 계산 (믿을 수 있는 값)
    - scorer_coverage : v3가 놓친 구간은 앞뒤 위치를 직선으로 이어 정답 위치를 '추정'해서 포함
                        → 그 구간에서는 근처의 다른 선수가 잡힐 수 있으니 참고용(근사값)
    """
    ref_by_frame, _ = load_tracking(tracking_json_of(stem))       # 정답 경로는 항상 v3 기준
    run_by_frame, _ = load_tracking(run_json)
    known = {}
    for tid, (a, b) in cfg["scorer"]:
        for fid in range(a, b + 1):
            for p in ref_by_frame.get(fid, []):
                if p["track_id"] == tid:
                    known[fid] = p["feet"]
    if not known:
        print(f"⚠️ {stem}: v3 JSON에서 득점 선수 track_id를 찾지 못했습니다. EVAL_CLIPS의 scorer 값을 확인하세요.")
        return {"scorer_frames": 0, "scorer_coverage": 0.0, "scorer_ids": 0, "scorer_id_switches": 0}
    span = range(min(known), max(known) + 1)
    kf = np.array(sorted(known))
    kx = np.array([known[f][0] for f in kf]); ky = np.array([known[f][1] for f in kf])

    seq = []   # (frame, 찾은 track_id 또는 None, 정답 위치가 확실한지)
    for fid in span:
        gx, gy = (known[fid] if fid in known else (np.interp(fid, kf, kx), np.interp(fid, kf, ky)))
        best, bd = None, 1e9
        for p in run_by_frame.get(fid, []):
            d = np.hypot(p["feet"][0] - gx, p["feet"][1] - gy)
            if d < bd:
                best, bd = p, d
        tid = best["track_id"] if best is not None and bd <= SCORER_MATCH_PX else None
        seq.append((fid, tid, fid in known))

    sure = [t for _, t, k in seq if k and t is not None]
    switches = sum(1 for a, b in zip(sure, sure[1:]) if a != b)
    covered = sum(1 for _, t, _ in seq if t is not None)

    if verbose:   # 어느 ID가 어느 구간을 맡았는지 (v3 노트북의 SEGMENTS와 같은 형태) → 영상으로 확인할 때 사용
        runs, cur = [], None
        for fid, t, _ in seq:
            if cur and cur[0] == t and cur[2] == fid - 1:
                cur[2] = fid
            else:
                cur = [t, fid, fid]; runs.append(cur)
        desc = ", ".join(f"{'없음' if t is None else '#' + str(t)}({a}~{b})" for t, a, b in runs)
        print(f"  [{stem}] 득점 선수 경로: {desc}")

    return {
        "scorer_frames": len(seq),
        "scorer_coverage": round(covered / len(seq), 3),
        "scorer_ids": len(set(sure)),
        "scorer_id_switches": switches,
    }


def segment_stats(stem, run_json, cfg):
    """플레이 구간 안의 고유 ID 수, Unknown 비율 (라벨링 없이 측정)"""
    by_frame, _ = load_tracking(run_json)
    a, b = cfg["play_range"]
    ids, types = set(), Counter()
    for fid in range(a, b + 1):
        for p in by_frame.get(fid, []):
            ids.add(p["track_id"]); types[p["type"]] += 1
    total = sum(types.values())
    return {"segment_unique_ids": len(ids),
            "segment_unknown_ratio": round(types["Unknown"] / total, 3) if total else 0.0}


RESULTS_CSV = os.path.join(EVAL_DIR, "eval_results.csv")


def evaluate_run(run_name, run_json_by_stem, note=""):
    """
    한 번의 실험 결과를 채점해서 eval_results.csv에 한 줄 추가합니다.
    run_json_by_stem: {stem: 그 실험이 만든 tracking_final.json 경로}
    """
    det, conf_mat, per_frame = detection_and_team_metrics(run_json_by_stem)
    row = {"run": run_name, "note": note, **det}
    for stem, jpath in run_json_by_stem.items():
        cfg = EVAL_CLIPS[stem]
        k = cfg["clip_key"]
        for key, val in {**scorer_track_metrics(stem, jpath, cfg), **segment_stats(stem, jpath, cfg)}.items():
            row[f"{key}_{k}"] = val
    df_new = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        old = pd.read_csv(RESULTS_CSV)
        old = old[old["run"] != run_name]                      # 같은 이름으로 다시 돌리면 덮어쓰기
        df_new = pd.concat([old, df_new], ignore_index=True)
    df_new.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
    return row, conf_mat, per_frame


def show_confusion(conf_mat):
    rows = EVAL_TEAM + (["Unknown"] if any(g == "Unknown" for g, _ in conf_mat) else [])
    cols = CLASS_NAMES
    return pd.DataFrame([[conf_mat.get((r, c), 0) for c in cols] for r in rows],
                        index=[f"정답:{r}" for r in rows], columns=[f"예측:{c}" for c in cols])

print("✅ 평가 함수 준비 완료")

In [ ]:
# 틀린 곳을 눈으로 확인: 초록=맞게 찾음(TP), 빨강=놓친 정답(FN), 파랑=잘못 잡음(FP)
def show_errors(name, run_json_by_stem):
    stem = name.split("_f")[0]
    fid = int(name.split("_f")[-1])
    by_frame, poly = load_tracking(run_json_by_stem[stem])
    preds = by_frame.get(fid, [])
    g = GT[name]
    pairs, ug, up = match_by_center([b for _, b in g["boxes"]], [p["bbox"] for p in preds])
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    cv2.polylines(img, [poly.reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    for gi, pi in pairs:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 200, 0), 2)
    for gi in ug:
        b = g["boxes"][gi][1]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    for pi in up:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 2)
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{name}  TP={len(pairs)}  FN={len(ug)}  FP={len(up)}"); plt.axis("off"); plt.show()

## ByteTrack
v3는 Ultralytics 안의 ByteTrack을 썼어요. 여기서는 저장해둔 탐지 결과에 설정만 바꿔 여러 번 돌리려고,
같은 규칙을 numpy로 옮긴 버전을 씁니다. 셀 끝의 자체 점검이 통과하는지 확인하세요.

In [ ]:
# ===== ByteTrack (논문 · Ultralytics 구현과 같은 규칙을 그대로 옮긴 버전) =====
# 저장해둔 탐지 결과에 바로 돌릴 수 있도록 numpy만으로 작성했습니다.
#   1) 칼만 필터로 기존 트랙의 이번 위치 예측
#   2) 높은 점수 박스와 IoU 짝짓기 (헝가리안)
#   3) 짝 못 찾은 트랙은 낮은 점수 박스와 한 번 더 짝짓기  ← ByteTrack의 핵심
#   4) 남은 높은 점수 박스 중 new_track_thresh 이상만 새 트랙으로 등록
#   5) track_buffer 프레임 동안 못 찾은 트랙은 삭제

class KalmanXYAH:
    """상태: [cx, cy, 종횡비 a, 높이 h, 각 속도 4개]. 등속 모델."""
    def __init__(self):
        self.F = np.eye(8)
        for i in range(4):
            self.F[i, 4 + i] = 1.0
        self.H = np.eye(4, 8)
        self.wp, self.wv = 1.0 / 20, 1.0 / 160

    def initiate(self, z):
        h = z[3]
        mean = np.r_[z, np.zeros(4)]
        std = [2 * self.wp * h, 2 * self.wp * h, 1e-2, 2 * self.wp * h,
               10 * self.wv * h, 10 * self.wv * h, 1e-5, 10 * self.wv * h]
        return mean, np.diag(np.square(std))

    def predict(self, mean, cov):
        h = mean[3]
        q = [self.wp * h, self.wp * h, 1e-2, self.wp * h, self.wv * h, self.wv * h, 1e-5, self.wv * h]
        return self.F @ mean, self.F @ cov @ self.F.T + np.diag(np.square(q))

    def update(self, mean, cov, z):
        h = mean[3]
        R = np.diag(np.square([self.wp * h, self.wp * h, 1e-1, self.wp * h]))
        S = self.H @ cov @ self.H.T + R
        K = np.linalg.solve(S, (cov @ self.H.T).T).T
        mean = mean + K @ (z - self.H @ mean)
        cov = cov - K @ S @ K.T
        return mean, cov


def xyxy_to_xyah(b):
    w, h = b[2] - b[0], b[3] - b[1]
    return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, w / max(h, 1e-6), h], float)


def xyah_to_xyxy(m):
    cx, cy, a, h = m[:4]
    w = a * h
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], float)


def iou_matrix_np(a, b):
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None); inter = wh[..., 0] * wh[..., 1]
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def linear_assign(cost, thresh):
    if cost.size == 0:
        return [], list(range(cost.shape[0])), list(range(cost.shape[1]))
    c = np.where(cost <= thresh, cost, 1e6)
    r, k = linear_sum_assignment(c)
    m = [(i, j) for i, j in zip(r, k) if c[i, j] <= thresh]
    mi = {i for i, _ in m}; mj = {j for _, j in m}
    return m, [i for i in range(cost.shape[0]) if i not in mi], [j for j in range(cost.shape[1]) if j not in mj]


class Track:
    _next_id = 0
    TRACKED, LOST, REMOVED = 0, 1, 2

    def __init__(self, box, score, det_idx, kf):
        self.kf, self.box, self.score, self.det_idx = kf, np.asarray(box, float), score, det_idx
        self.mean = self.cov = None
        self.state, self.activated, self.track_id = Track.TRACKED, False, -1
        self.start_frame = self.end_frame = 0

    def predicted_box(self):
        return xyah_to_xyxy(self.mean) if self.mean is not None else self.box

    def predict(self):
        m = self.mean.copy()
        if self.state != Track.TRACKED:
            m[7] = 0.0                          # 놓친 트랙은 높이 변화 속도 0
        self.mean, self.cov = self.kf.predict(m, self.cov)

    def activate(self, frame_id):
        Track._next_id += 1
        self.track_id = Track._next_id
        self.mean, self.cov = self.kf.initiate(xyxy_to_xyah(self.box))
        self.state = Track.TRACKED
        self.activated = frame_id == 1
        self.start_frame = self.end_frame = frame_id

    def update(self, det, frame_id):
        self.mean, self.cov = self.kf.update(self.mean, self.cov, xyxy_to_xyah(det.box))
        self.box, self.score, self.det_idx = det.box, det.score, det.det_idx
        self.state, self.activated, self.end_frame = Track.TRACKED, True, frame_id


class ByteTracker:
    def __init__(self, track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6,
                 track_buffer=30, match_thresh=0.8, fuse_score=True, frame_rate=30):
        self.hi, self.lo, self.new = track_high_thresh, track_low_thresh, new_track_thresh
        self.max_lost = int(frame_rate / 30.0 * track_buffer)
        self.match, self.fuse = match_thresh, fuse_score
        self.kf = KalmanXYAH()
        self.tracked, self.lost = [], []
        self.frame_id = 0
        Track._next_id = 0

    def _dist(self, tracks, dets, fuse):
        d = 1 - iou_matrix_np([t.predicted_box() for t in tracks], [x.box for x in dets])
        if fuse and len(dets):
            d = 1 - (1 - d) * np.array([x.score for x in dets])[None, :]
        return d

    def update(self, boxes, scores):
        """boxes (N,4) xyxy, scores (N,) → [(track_id, det_idx), ...] 이번 프레임에 확정된 트랙"""
        self.frame_id += 1
        f = self.frame_id
        boxes = np.asarray(boxes, float).reshape(-1, 4); scores = np.asarray(scores, float).reshape(-1)
        high = [Track(boxes[i], scores[i], i, self.kf) for i in np.where(scores >= self.hi)[0]]
        low = [Track(boxes[i], scores[i], i, self.kf) for i in np.where((scores > self.lo) & (scores < self.hi))[0]]

        unconfirmed = [t for t in self.tracked if not t.activated]
        confirmed = [t for t in self.tracked if t.activated]
        pool = confirmed + [t for t in self.lost if t not in confirmed]
        for t in pool:
            t.predict()

        refound, activated_now = [], []
        # 1차: 높은 점수 박스
        m, u_t, u_d = linear_assign(self._dist(pool, high, self.fuse), self.match)
        for i, j in m:
            t = pool[i]
            if t.state != Track.TRACKED:
                refound.append(t)
            t.update(high[j], f); activated_now.append(t)
        # 2차: 남은 '추적 중' 트랙 × 낮은 점수 박스
        r_tracked = [pool[i] for i in u_t if pool[i].state == Track.TRACKED]
        m2, u_t2, _ = linear_assign(self._dist(r_tracked, low, False), 0.5)
        for i, j in m2:
            r_tracked[i].update(low[j], f); activated_now.append(r_tracked[i])
        for i in u_t2:
            r_tracked[i].state = Track.LOST
        # 확정 안 된(1프레임짜리) 트랙 × 남은 높은 점수 박스
        rest_high = [high[j] for j in u_d]
        m3, u_un, u_d3 = linear_assign(self._dist(unconfirmed, rest_high, self.fuse), 0.7)
        for i, j in m3:
            unconfirmed[i].update(rest_high[j], f); activated_now.append(unconfirmed[i])
        for i in u_un:
            unconfirmed[i].state = Track.REMOVED
        # 새 트랙
        new_tracks = []
        for j in u_d3:
            d = rest_high[j]
            if d.score >= self.new:
                d.activate(f); new_tracks.append(d)
        # 오래 못 찾은 트랙 삭제
        for t in pool:
            if t.state == Track.LOST and f - t.end_frame > self.max_lost:
                t.state = Track.REMOVED

        everyone = {id(t): t for t in self.tracked + self.lost + new_tracks}
        self.tracked = [t for t in everyone.values() if t.state == Track.TRACKED]
        self.lost = [t for t in everyone.values() if t.state == Track.LOST]
        return [(t.track_id, t.det_idx) for t in self.tracked if t.activated and t.end_frame == f]


def _bytetrack_selftest():
    """작은 합성 장면으로 동작 확인: 점수가 잠깐 낮아지거나 몇 프레임 사라져도 ID 유지, 약한 새 박스는 새 ID 안 받음"""
    tr = ByteTracker(track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6, track_buffer=30)
    ids = defaultdict(set)
    for f in range(1, 61):
        boxes, scores = [], []
        x = 100 + 3 * f                                     # 선수 A: 오른쪽으로 이동
        if not (30 <= f < 36):                              # 30~35프레임 동안 사라짐(가려짐)
            boxes.append([x, 100, x + 12, 130]); scores.append(0.3 if 15 <= f < 20 else 0.8)   # 15~19 낮은 점수
        boxes.append([300, 200 + f, 312, 230 + f]); scores.append(0.9)                      # 선수 B
        if f == 40:
            boxes.append([500, 50, 512, 80]); scores.append(0.4)                            # 약한 오탐 1번
        for tid, di in tr.update(boxes, scores):
            ids["A" if boxes[di][0] < 290 else ("B" if boxes[di][0] < 400 else "X")].add(tid)
    assert len(ids["A"]) == 1 and len(ids["B"]) == 1 and "X" not in ids, dict(ids)
    print("✅ ByteTrack 자체 점검 통과:", {k: sorted(v) for k, v in ids.items()})

_bytetrack_selftest()

In [ ]:
# ===== 자동 ROI: 참조 프레임 → 새 프레임으로 경기장 영역 옮기기 =====
# 같은 경기장이면 '경기장 영역을 이미 아는 프레임(참조)'과 새 프레임은 같은 평면(경기장)을 다른 위치·각도에서 본 것.
# 평면 → 평면의 원근 변환은 3×3 호모그래피 H 하나로 표현됨:  [x', y', w']ᵀ = H [x, y, 1]ᵀ,  (x'/w', y'/w')
#   1) SIFT 특징점 추출 (크기·회전·밝기 변화에 강함) — 반 해상도에서 계산해 속도 확보
#   2) 참조 ↔ 새 프레임 특징점 짝짓기 + Lowe 비율 검사(0.75)로 애매한 짝 제거
#   3) RANSAC으로 호모그래피 추정: 무작위 4쌍으로 H를 만들고, 5px 안에 맞는 짝(인라이어)이 가장 많은 H 선택
#      → 나무·자동차처럼 경기장 평면이 아닌 점, 잘못된 짝은 자동으로 버려짐
#   4) 참조 ROI 다각형을 H로 옮김. 참조 화면 가장자리에서 잘린 변은 실제 라인 방향으로 화면 밖까지 연장한 뒤 옮김
#      (새 프레임이 더 넓게 찍혔을 때 ROI가 참조 화면 크기에서 잘리지 않도록)
#   5) 인라이어가 MIN_INLIERS 미만이면 '경기장의 다른 쪽을 찍은 영상'으로 보고 실패 처리 (수동 입력 필요)
import time
SIFT = cv2.SIFT_create(nfeatures=6000)
BF = cv2.BFMatcher()
FEAT_SCALE = 0.5
MIN_INLIERS = 100
RATIO = 0.75


def sift_features(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, None, fx=FEAT_SCALE, fy=FEAT_SCALE, interpolation=cv2.INTER_AREA)
    kp, des = SIFT.detectAndCompute(g, None)
    return np.float32([k.pt for k in kp]).reshape(-1, 2) / FEAT_SCALE, des


def _border_sides(p, W, H, tol=3):
    x, y = p
    return {s for s, c in (("L", x <= tol), ("R", x >= W - 1 - tol), ("T", y <= tol), ("B", y >= H - 1 - tol)) if c}


def open_polygon(poly, W, H, L=2500):
    """화면 가장자리에서 잘린 ROI 꼭짓점을 실제 라인 방향으로 L px 연장 (화면 모서리 꼭짓점은 버림)"""
    P = [np.float64(p) for p in poly]; out = []
    for i, p in enumerate(P):
        a, b = P[i - 1], P[(i + 1) % len(P)]
        bp = _border_sides(p, W, H)
        if not bp:
            out.append(p); continue
        ea, eb = bool(bp & _border_sides(a, W, H)), bool(bp & _border_sides(b, W, H))   # 이웃과 같은 가장자리 = 화면 경계 변
        if ea and eb:
            continue
        q = b if ea else a                                                              # 실제 라인 쪽 이웃
        d = (p - q) / (np.linalg.norm(p - q) + 1e-9)
        out.append(p + d * L)
    return np.array(out)


def make_ref(name, img, roi_pts):
    H, W = img.shape[:2]
    pts, des = sift_features(img)
    return {"name": name, "pts": pts, "des": des, "poly_open": open_polygon(roi_pts, W, H), "roi": np.asarray(roi_pts)}


def estimate_transform(ref, pts, des, model="homography"):
    """참조 → 새 프레임 변환과 인라이어 수. model: 'homography'(원근, 8자유도) / 'similarity'(이동·회전·크기, 4자유도)"""
    if des is None or ref["des"] is None or len(des) < 10:
        return None, 0
    good = [p[0] for p in BF.knnMatch(ref["des"], des, k=2) if len(p) == 2 and p[0].distance < RATIO * p[1].distance]
    if len(good) < 10:
        return None, 0
    a = ref["pts"][[m.queryIdx for m in good]]; b = pts[[m.trainIdx for m in good]]
    if model == "homography":
        Hm, inl = cv2.findHomography(a, b, cv2.RANSAC, 5.0)
    else:
        A, inl = cv2.estimateAffinePartial2D(a, b, method=cv2.RANSAC, ransacReprojThreshold=5.0)
        Hm = None if A is None else np.vstack([A, [0, 0, 1]])
    return (None, 0) if Hm is None else (Hm, int(inl.sum()))


def warp_roi_mask(ref, Hm, W, H):
    pts = np.hstack([ref["poly_open"], np.ones((len(ref["poly_open"]), 1))]) @ Hm.T
    if np.any(pts[:, 2] <= 1e-6):                       # 연장한 점이 지평선 너머로 넘어감 → 실패
        return None
    m = np.zeros((H, W), np.uint8)
    cv2.fillPoly(m, [np.round(pts[:, :2] / pts[:, 2:]).astype(np.int32).reshape(-1, 1, 2)], 1)
    return m


def mask_to_poly(m):
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.approxPolyDP(max(cnts, key=cv2.contourArea), 2.0, True).reshape(-1, 2)


def auto_roi(img, refs, model="homography", exclude=()):
    """→ (ROI 다각형 또는 None, 고른 참조 이름, 인라이어 수)"""
    H, W = img.shape[:2]
    pts, des = sift_features(img)
    best = None
    for ref in refs:
        if ref["name"] in exclude:
            continue
        Hm, n = estimate_transform(ref, pts, des, model)
        if Hm is not None and n >= MIN_INLIERS and (best is None or n > best[2]):
            best = (ref, Hm, n)
    if best is None:
        return None, None, 0
    m = warp_roi_mask(best[0], best[1], W, H)
    if m is None or m.sum() < 0.05 * W * H:             # 화면의 5%도 안 되면 잘못 옮긴 것으로 봄
        return None, best[0]["name"], best[2]
    return mask_to_poly(m), best[0]["name"], best[2]


def expand_polygon(poly, margin, W, H):
    """다각형을 margin(px)만큼 바깥으로 넓힘 (1a와 같은 방법: 채운 마스크 팽창 → 다시 다각형)"""
    mask = np.zeros((H, W), np.uint8)
    cv2.fillPoly(mask, [np.asarray(poly, np.int32).reshape(-1, 1, 2)], 255)
    if margin > 0:
        mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * margin + 1, 2 * margin + 1)))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.approxPolyDP(max(cnts, key=cv2.contourArea), 2.0, True).reshape(-1, 2)


def poly_iou(p1, p2, W, H):
    a = np.zeros((H, W), np.uint8); b = np.zeros((H, W), np.uint8)
    cv2.fillPoly(a, [np.int32(p1).reshape(-1, 1, 2)], 1); cv2.fillPoly(b, [np.int32(p2).reshape(-1, 1, 2)], 1)
    return float((a & b).sum() / max(1, (a | b).sum()))

print("✅ 자동 ROI 함수 준비 완료 (OpenCV", cv2.__version__, ")")

---
# 6a. 추적 · 팀 라벨 안정화

| 순서 | 내용 | 런타임 |
|---|---|---|
| ① | 파인튜닝 YOLO로 평가 클립 2개 전체를 탐지해 **캐시로 저장** (한 번만) | **GPU 약 7분** |
| ② | 팀 라벨 정하는 방식 4가지 비교 → 채점 | CPU 몇 분 |
| ③ | ROI 갱신 안정성 검사 (직전과 너무 다르면 버리기) → 프레임별 인원으로 확인 | CPU 약 5분 |
| ④ | 득점 선수 정답 경로가 맞는지 눈으로 확인 | CPU 1분 |

- ①이 끝나면 런타임을 끊고 **CPU 런타임으로 다시 열어** ②부터 해도 돼요 (캐시가 Drive에 저장돼요).
- 비교 방식은 **결과를 보기 전에 정해둔 것**(구간 15프레임)을 대표로 쓰고, 다른 구간 길이는 참고로만 봐요. 평가셋을 보고 설정을 고르면 안 되니까요 (1단원).

In [ ]:
import time, pickle
FT_WEIGHTS = os.path.join(EVAL_DIR, "finetune", "yolo26_ft_best.pt")
CACHE_DIR = os.path.join(EVAL_DIR, "cache"); os.makedirs(CACHE_DIR, exist_ok=True)
TEAM = ["Hongik", "Kookmin", "Referee"]
TRACK_PARAMS = json.load(open(os.path.join(EVAL_DIR, "tracker_config_v4.json"), encoding="utf-8"))["params"]
WARMUP = 60
WIN_MAIN = 15            # 사전에 정한 구간: 앞뒤 15프레임(0.5초)
ROI_IOU_MIN = 0.85       # 새 ROI가 직전 ROI와 이보다 덜 겹치면 버림 (드론은 1초에 크게 안 움직임)
ROI_1A = {}
for stem in EVAL_CLIPS:
    _, ROI_1A[stem] = load_tracking(os.path.join(EVAL_DIR, "runs", "step1a_roi", f"{stem}_tracking_final.json"))
cache_path = lambda stem: os.path.join(CACHE_DIR, f"{stem}_ft_full.pkl")
assert os.path.exists(FT_WEIGHTS), f"파인튜닝 가중치가 없습니다: {FT_WEIGHTS} (5b를 먼저 실행)"

In [ ]:
# ① 파인튜닝 YOLO 탐지 캐시 (GPU). 이미 있으면 건너뜀
def build_cache(stem):
    import torch
    from ultralytics import YOLO
    dev = 0 if torch.cuda.is_available() else "cpu"
    assert dev == 0, "이 셀은 GPU가 필요해요 (런타임 → 런타임 유형 변경 → T4 GPU)"
    m = YOLO(FT_WEIGHTS)
    cap = cv2.VideoCapture(video_path_of(stem)); fid, frames, t0 = 0, {}, time.time()
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    while True:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1
        r = m.predict(img, imgsz=1920, conf=0.05, max_det=300, device=dev, verbose=False)[0]
        frames[fid] = {"boxes": r.boxes.xyxy.cpu().numpy().reshape(-1, 4), "scores": r.boxes.conf.cpu().numpy().reshape(-1),
                       "cls": r.boxes.cls.cpu().numpy().astype(int).reshape(-1)}
        if fid % 300 == 0:
            print(f"  [{stem[-6:-2]}] {fid}/{n}프레임 · {time.time() - t0:.0f}s")
    cap.release()
    c = {"start": 1, "end": fid, "frames": frames}
    pickle.dump(c, open(cache_path(stem), "wb"))
    return c

for stem in EVAL_CLIPS:
    if os.path.exists(cache_path(stem)):
        print(f"⏭️ 캐시 있음: {stem}")
    else:
        print(f"▶ {stem} 탐지 시작"); build_cache(stem); print(f"✅ 저장 → {cache_path(stem)}")
print("\n① 끝. GPU를 아끼려면 지금 런타임을 끊고, CPU 런타임으로 공통 셀부터 다시 실행한 뒤 ②로 오세요.")

In [ ]:
# ② 팀 라벨 방식 비교
CACHE = {s: pickle.load(open(cache_path(s), "rb")) for s in EVAL_CLIPS}


def track_clip(cache, roi_at, f_start, f_end):
    """캐시된 탐지 → ROI(발 위치) 필터 → ByteTrack. → {fid: [(tid, box, score, cls)]}"""
    tr = ByteTracker(**TRACK_PARAMS, fuse_score=True, frame_rate=30); out = {}
    for fid in range(f_start, f_end + 1):
        fr = cache["frames"].get(fid)
        if fr is None:
            out[fid] = []; continue
        poly = roi_at(fid)
        k = [i for i, b in enumerate(fr["boxes"]) if inside(poly, (b[0] + b[2]) / 2, b[3])]
        bx, sc, cl = fr["boxes"][k], fr["scores"][k], fr["cls"][k]
        out[fid] = [(int(t), bx[d], float(sc[d]), int(cl[d])) for t, d in tr.update(bx, sc)]
    return out


def by_track(rows):
    T = defaultdict(list)
    for fid in sorted(rows):
        for tid, b, s, c in rows[fid]:
            T[tid].append((fid, c, s))
    return T


def label_track_majority(rows):
    """v6 방식: 트랙 전체를 점수 가중 다수결로 한 팀"""
    lab = {}
    for tid, seq in by_track(rows).items():
        v = Counter()
        for _, c, s in seq:
            v[c] += s
        best = v.most_common(1)[0][0]
        for fid, _, _ in seq:
            lab[(tid, fid)] = best
    return lab


def label_per_box(rows):
    return {(tid, fid): c for fid in rows for tid, b, s, c in rows[fid]}


def label_window(rows, k):
    """트랙 안에서 앞뒤 k프레임의 점수 가중 다수결"""
    lab = {}
    for tid, seq in by_track(rows).items():
        f = np.array([x[0] for x in seq]); c = np.array([x[1] for x in seq]); s = np.array([x[2] for x in seq])
        lo = np.searchsorted(f, f - k, "left"); hi = np.searchsorted(f, f + k, "right")
        for i in range(len(seq)):
            w = np.bincount(c[lo[i]:hi[i]], weights=s[lo[i]:hi[i]], minlength=len(TEAM))
            lab[(tid, int(f[i]))] = int(w.argmax())
    return lab


def split_on_team_change(rows, lab):
    """구간 다수결 라벨이 바뀌는 지점에서 트랙을 새 ID로 자름 (다른 팀 선수에게 넘어간 ID 분리)"""
    seg_of = {}
    for tid, seq in by_track(rows).items():
        seg, prev = 0, None
        for fid, _, _ in seq:
            l = lab[(tid, fid)]
            if prev is not None and l != prev:
                seg += 1
            prev = l; seg_of[(tid, fid)] = seg
    new_id, new_rows, new_lab = {}, {}, {}
    for fid in sorted(rows):
        rr = []
        for tid, b, s, c in rows[fid]:
            key = (tid, seg_of[(tid, fid)])
            nid = new_id.setdefault(key, len(new_id) + 1)
            rr.append((nid, b, s, c)); new_lab[(nid, fid)] = lab[(tid, fid)]
        new_rows[fid] = rr
    return new_rows, new_lab


def write_run(run_name, stem, rows, lab, poly, note):
    frames = []
    for fid in sorted(rows):
        ps = []
        for tid, b, s, c in rows[fid]:
            x1, y1, x2, y2 = [int(round(v)) for v in b]
            ps.append({"track_id": tid, "type": TEAM[lab[(tid, fid)]], "bbox": [x1, y1, x2, y2], "feet": [(x1 + x2) // 2, y2], "conf": s})
        frames.append({"frame_id": fid, "players": ps})
    p = os.path.join(EVAL_DIR, "runs", run_name, f"{stem}_tracking_final.json")
    os.makedirs(os.path.dirname(p), exist_ok=True)
    json.dump({"video_name": os.path.basename(video_path_of(stem)), "fps": 30, "field_polygon": np.asarray(poly).tolist(),
               "frames": frames, "notes": [note]}, open(p, "w", encoding="utf-8"), ensure_ascii=False)
    return p


# v6와 같은 조건: 1a ROI 고정, 플레이 시작 60프레임 전부터 끝까지
ROWS = {}
for stem, cfg in EVAL_CLIPS.items():
    a, b = cfg["play_range"]
    ROWS[stem] = track_clip(CACHE[stem], lambda f, s=stem: ROI_1A[s], max(1, a - WARMUP), b)

VARIANTS = {
    "v6c_track_majority": ("트랙 전체 다수결 (v6 재현)", lambda r: (r, label_track_majority(r))),
    "v6c_per_box":        ("박스마다 YOLO 클래스", lambda r: (r, label_per_box(r))),
    "v6c_win15":          (f"앞뒤 {WIN_MAIN}프레임 다수결 (대표)", lambda r: (r, label_window(r, WIN_MAIN))),
    "v6c_win15_split":    (f"앞뒤 {WIN_MAIN}프레임 다수결 + 팀 바뀌면 트랙 자르기", lambda r: split_on_team_change(r, label_window(r, WIN_MAIN))),
}
CONFS = {}
for run, (note, fn) in VARIANTS.items():
    paths = {}
    for stem in EVAL_CLIPS:
        r2, lab = fn(ROWS[stem])
        paths[stem] = write_run(run, stem, r2, lab, ROI_1A[stem], f"6a: 파인튜닝 YOLO 캐시 + 2a 추적 + {note}")
    _, CONFS[run], _ = evaluate_run(run, paths, note=f"6a: {note}")

res = pd.read_csv(RESULTS_CSV).set_index("run")
cols = ["det_f1", "TP", "FP", "FN", "team_acc", "referee_recall", "segment_unique_ids_34", "segment_unique_ids_44"]
order = [r for r in ["v5_cnn", "v6_yolo_ft"] + list(VARIANTS) if r in res.index]
display(res.loc[order, cols])
print("v6c_track_majority가 v6_yolo_ft와 거의 같으면 캐시가 제대로 만들어진 것")
for run in ["v6c_track_majority", "v6c_win15"]:
    print(f"\n=== {run} 혼동행렬 ==="); display(show_confusion(CONFS[run]))

# 참고: 구간 길이에 따른 변화 (대표값은 15로 미리 정함. 이 표를 보고 바꾸지 않기)
sens = {}
for k in (5, 15, 30, 60):
    accs = []
    for stem in EVAL_CLIPS:
        write_run("tmp_sens", stem, ROWS[stem], label_window(ROWS[stem], k), ROI_1A[stem], "tmp")
    row, _, _ = evaluate_run("tmp_sens", {s: os.path.join(EVAL_DIR, "runs", "tmp_sens", f"{s}_tracking_final.json") for s in EVAL_CLIPS}, note="tmp")
    sens[f"앞뒤 {k}프레임"] = {"team_acc": row["team_acc"], "referee_recall": row["referee_recall"]}
_r = pd.read_csv(RESULTS_CSV); _r[_r["run"] != "tmp_sens"].to_csv(RESULTS_CSV, index=False)   # 참고용 행은 결과표에서 지움
display(pd.DataFrame(sens).T)

In [ ]:
# ③ ROI 갱신 안정성: 클립 전체에서 30프레임마다 자동 ROI, 검사 있음/없음 비교
KNOWN = {
    "DJI_20250913133612_0144_D": (413, [[245, 530], [1860, 543], [1919, 776], [1919, 1079], [107, 1079]]),
    "DJI_20250913135020_0156_D": (167, [[0, 0], [1632, 0], [1712, 810], [0, 787]]),
    "DJI_20250913133530_0143_D": (1, [[270, 459], [1880, 469], [1919, 612], [1919, 1079], [119, 1079]]),
    "DJI_20250913135111_0157_D": (1, [[0, 0], [1675, 0], [1761, 737], [0, 733]]),
}
ROI_MARGIN_PX, ROI_UPDATE_EVERY = 5, 30


def read_frame(stem, fid):
    cap = cv2.VideoCapture(video_path_of(stem)); img = None
    for _ in range(fid):
        ok, img = cap.read()
    cap.release()
    return img

REFS = [make_ref(s, read_frame(s, f), roi) for s, (f, roi) in KNOWN.items()]


def roi_candidates(stem):
    """30프레임마다 자동 ROI 후보 계산 (SIFT는 한 번만) → {갱신 프레임: (ROI 또는 None, 참조, 인라이어)}, 마지막 프레임 번호"""
    cap = cv2.VideoCapture(video_path_of(stem)); fid, cand = 0, {}
    while True:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1
        if (fid - 1) % ROI_UPDATE_EVERY == 0:
            H, W = img.shape[:2]
            poly, ref, n = auto_roi(img, REFS)
            cand[fid] = (expand_polygon(poly, ROI_MARGIN_PX, W, H) if poly is not None else None, ref, n, (W, H))
    cap.release()
    return cand, fid


def roi_sequence(cand, last, check, fallback):
    """check=True면 직전 ROI와 IoU < ROI_IOU_MIN인 새 ROI는 버리고 직전 유지"""
    cur, seq, log = None, {}, []
    for fid in range(1, last + 1):
        if fid in cand and cand[fid][0] is not None:
            new, ref, n, (W, H) = cand[fid]
            iou = poly_iou(new, cur, W, H) if cur is not None else 1.0
            if check and cur is not None and iou < ROI_IOU_MIN:
                log.append({"frame": fid, "직전과 IoU": round(iou, 3), "참조": ref[-6:-2], "인라이어": n, "결과": "버림"})
            else:
                if iou < 0.95:
                    log.append({"frame": fid, "직전과 IoU": round(iou, 3), "참조": ref[-6:-2], "인라이어": n, "결과": "적용"})
                cur = new
        seq[fid] = cur if cur is not None else fallback
    return seq, pd.DataFrame(log)


def team_counts(rows, lab):
    return pd.DataFrame([Counter(TEAM[lab[(t, f)]] for t, b, s, c in rows[f]) for f in sorted(rows)], index=sorted(rows)).reindex(columns=TEAM).fillna(0)


ROI_STUDY = {}
for stem in EVAL_CLIPS:
    t0 = time.time()
    cand, last = roi_candidates(stem)
    last = min(last, CACHE[stem]["end"])
    print(f"{stem[-6:-2]}: 자동 ROI 후보 {len(cand)}개 계산 ({time.time() - t0:.0f}s), 실패 {sum(v[0] is None for v in cand.values())}개")
    for check in (False, True):
        seq, log = roi_sequence(cand, last, check, ROI_1A[stem])
        rows = track_clip(CACHE[stem], lambda f, q=seq: q[f], 1, last)
        cnt = team_counts(rows, label_window(rows, WIN_MAIN))
        ROI_STUDY[(stem, check)] = cnt
        name = "검사 있음" if check else "검사 없음"
        print(f"  · {name}: 평균 인원 {cnt.mean().round(1).to_dict()} · 한 팀이 14명 넘은 프레임 {int((cnt.max(axis=1) > 14).sum())}개")
        if len(log):
            print(f"    ROI가 크게 바뀐 갱신 ({name}):"); display(log)

fig, axes = plt.subplots(len(EVAL_CLIPS), 1, figsize=(14, 3.2 * len(EVAL_CLIPS)), squeeze=False)
for ax, stem in zip(axes[:, 0], EVAL_CLIPS):
    for check, ls in ((False, ":"), (True, "-")):
        c = ROI_STUDY[(stem, check)]
        for team, col in (("Hongik", "tab:blue"), ("Kookmin", "tab:red")):
            ax.plot(c.index, c[team].rolling(15, min_periods=1).mean(), ls, color=col, label=f"{team} ({'검사 있음' if check else '검사 없음'})")
    ax.axhline(11, color="gray", lw=1); ax.set_title(f"{stem[-6:-2]} · 프레임별 팀 인원 (점선 = 검사 없음, 실선 = 검사 있음)")
    ax.legend(ncol=4, fontsize=8, loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ④ 득점 선수 정답 경로 확인: v3에서 이어 붙인 구간 경계에서 앞 트랙 · 뒤 트랙을 나란히
v3_by_f = {s: load_tracking(os.path.join(DRIVE_OUTPUT_DIR, f"{s}_tracking_final.json"))[0] for s in EVAL_CLIPS}


def box_of(stem, fid, tid):
    for p in v3_by_f[stem].get(fid, []):
        if p["track_id"] == tid:
            return p["bbox"]
    return None


def scorer_boundaries(stem):
    segs = EVAL_CLIPS[stem]["scorer"]
    out = []
    for (t1, (a1, b1)), (t2, (a2, b2)) in zip(segs, segs[1:]):
        out.append((t1, b1, t2, a2))
    return out


for stem in EVAL_CLIPS:
    bounds = scorer_boundaries(stem)
    if not bounds:
        print(f"{stem[-6:-2]}: 득점 선수 경로가 트랙 하나라 확인할 경계 없음"); continue
    need = sorted({f for t1, f1, t2, f2 in bounds for f in (f1 - 10, f1, f2, f2 + 10)})
    imgs = read_frames_sequential(video_path_of(stem), need)
    fig, axes = plt.subplots(len(bounds), 4, figsize=(14, 3.6 * len(bounds)), squeeze=False)
    for row, (t1, f1, t2, f2) in zip(axes, bounds):
        for ax, (f, t, tag) in zip(row, [(f1 - 10, t1, "앞 트랙"), (f1, t1, "앞 트랙 끝"), (f2, t2, "뒤 트랙 시작"), (f2 + 10, t2, "뒤 트랙")]):
            img, b = imgs.get(f), box_of(stem, f, t)
            if img is None or b is None:
                ax.set_title(f"{tag} #{t} · f{f}\n(박스 없음)", fontsize=9); ax.axis("off"); continue
            cx, cy = (b[0] + b[2]) // 2, (b[1] + b[3]) // 2
            x0, y0 = max(0, cx - 90), max(0, cy - 90)
            crop = img[y0:cy + 90, x0:cx + 90].copy()
            cv2.rectangle(crop, (b[0] - x0, b[1] - y0), (b[2] - x0, b[3] - y0), (0, 255, 255), 2)
            ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax.set_title(f"{tag} #{t} · f{f}", fontsize=9); ax.axis("off")
    fig.suptitle(f"{stem[-6:-2]} 득점 선수 정답 경로의 이음매 (노란 박스가 같은 선수인지 확인)"); plt.tight_layout(); plt.show()

## 결과 공유
1. ②의 비교표, 혼동행렬 2개, 구간 길이 참고표
2. ③의 클립별 요약 줄, 버린 ROI 기록 표, 인원 그래프
3. ④의 이음매 그림: 줄마다 **노란 박스가 같은 선수인지** (같음 / 다름 / 모르겠음)

---
## 득점 선수 정답 경로 다시 만들기 + ROI 스파이크 확인
④에서 0156의 예전 정답 경로는 이음매 3곳이 모두 **다른 선수**로 확인됐어요 (실제 득점자는 적갈색, 0144는 흰색).
그래서 정답을 **트랙 번호가 아니라 위치**로 다시 만들어요. 득점 구간에서 20프레임마다 득점자의 발 위치를 정답으로 삼아요.
위치로 저장하면 어떤 실행 결과든 같은 정답으로 채점할 수 있어요.

| 순서 | 내용 |
|---|---|
| ⑤ | 20프레임마다 화면에 모든 선수 번호를 붙여 보여줌 → 득점자 번호를 적을 빈 표가 출력됨 |
| ⑥ | 적은 번호 → 발 위치 정답 저장 → 실행 결과마다 득점자 추적 채점 |
| ⑦ | 0144의 1300프레임 근처 인원 급증이 ROI 때문인지 눈으로 확인 |

In [ ]:
# ⑤ 득점 구간 20프레임마다 모든 선수에 번호 붙여 보기
SCORER_COLOR = {"DJI_20250913133612_0144_D": "흰색 유니폼", "DJI_20250913135020_0156_D": "적갈색 유니폼"}
KEY_STEP = 20
SPLIT = {s: split_on_team_change(ROWS[s], label_window(ROWS[s], WIN_MAIN)) for s in EVAL_CLIPS}   # (rows, lab)
KEYS = {}
for s, cfg in EVAL_CLIPS.items():
    a, b = cfg["play_range"]
    KEYS[s] = sorted(set(range(a, b + 1, KEY_STEP)) | {b})

BOX_COLOR = {0: (255, 140, 0), 1: (40, 40, 230), 2: (0, 220, 255)}   # BGR: 홍익 파랑 · 국민 빨강 · 심판 노랑


def draw_ids(img, rows_f, lab, fid):
    out = img.copy()
    for tid, b, s, c in rows_f:
        x1, y1, x2, y2 = [int(v) for v in b]
        col = BOX_COLOR[lab[(tid, fid)]]
        cv2.rectangle(out, (x1, y1), (x2, y2), col, 2)
        for th, cc in ((5, (0, 0, 0)), (2, (255, 255, 255))):
            cv2.putText(out, str(tid), (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, cc, th)
    return out


for stem in EVAL_CLIPS:
    rows, lab = SPLIT[stem]
    imgs = read_frames_sequential(video_path_of(stem), KEYS[stem])
    print(f"\n===== {stem[-6:-2]} · 득점자 = {SCORER_COLOR[stem]} · {len(KEYS[stem])}장 =====")
    for fid in KEYS[stem]:
        plt.figure(figsize=(18, 10.2))
        plt.imshow(cv2.cvtColor(draw_ids(imgs[fid], rows.get(fid, []), lab, fid), cv2.COLOR_BGR2RGB))
        plt.title(f"{stem[-6:-2]} · f{fid} · 득점자({SCORER_COLOR[stem]})의 번호를 찾으세요", fontsize=14); plt.axis("off")
        plt.tight_layout(); plt.show()

print("\n아래 표를 복사해 ⑥ 셀 맨 위 SCORER_PICK에 붙여넣고 None을 번호로 바꾸세요.")
print("  번호 = 득점자 박스의 번호 · 0 = 득점자는 보이는데 박스가 없음 · -1 = 화면에 없거나 판단 불가")
print("SCORER_PICK = {")
for stem in EVAL_CLIPS:
    print(f'    "{stem}": {{' + ", ".join(f"{f}: None" for f in KEYS[stem]) + "},")
print("}")

In [ ]:
# ⑥ 적은 번호 → 발 위치 정답 저장 → 실행 결과마다 득점자 추적 채점
SCORER_PICK = {
    # ⑤에서 출력된 표를 여기에 붙여넣고 None을 번호로 바꾸세요
}
assert SCORER_PICK, "⑤가 출력한 표를 SCORER_PICK에 붙여넣으세요"
empty = [(s[-6:-2], f) for s, d in SCORER_PICK.items() for f, v in d.items() if v is None]
assert not empty, f"아직 None인 칸이 있어요: {empty[:10]}"

GT_PATH = os.path.join(EVAL_DIR, "scorer_gt_v2.json")
gt = {}
for stem, picks in SCORER_PICK.items():
    rows, lab = SPLIT[stem]; gt[stem] = {}
    for fid, tid in picks.items():
        if tid == -1:
            continue                                   # 판단 불가 → 채점에서 뺌
        if tid == 0:
            gt[stem][str(fid)] = None; continue        # 보이는데 박스 없음 → 놓친 것으로 채점
        hit = [b for t, b, s, c in rows.get(fid, []) if t == tid]
        assert hit, f"{stem[-6:-2]} f{fid}: 번호 {tid}인 박스가 없어요. 번호를 다시 확인하세요"
        b = hit[0]
        gt[stem][str(fid)] = [float((b[0] + b[2]) / 2), float(b[3])]
json.dump({"note": "6a ⑤⑥: 사람이 20프레임마다 확인한 득점자 발 위치 (None = 보이는데 박스 없음)", "gt": gt},
          open(GT_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(f"✅ 정답 저장 → {GT_PATH}")

MATCH_PX = 30   # 정답 발 위치와 이만큼 가까운 발 위치를 득점자로 봄


def scorer_score(run_json_path, gt_stem):
    by_f = {fr["frame_id"]: fr["players"] for fr in json.load(open(run_json_path, encoding="utf-8"))["frames"]}
    seq = []
    for f, pos in sorted(gt_stem.items(), key=lambda x: int(x[0])):
        if pos is None:
            seq.append(None); continue
        best, bd = None, MATCH_PX
        for p in by_f.get(int(f), []):
            d = float(np.hypot(p["feet"][0] - pos[0], p["feet"][1] - pos[1]))
            if d <= bd:
                best, bd = p["track_id"], d
        seq.append(best)
    hit = [t for t in seq if t is not None]
    switches = sum(1 for a, b in zip(hit, hit[1:]) if a != b)
    return {"정답칸": len(seq), "잡은 비율": round(len(hit) / max(1, len(seq)), 3),
            "ID 개수": len(set(hit)), "ID 바뀜": switches, "ID 순서": " → ".join("없음" if t is None else f"#{t}" for t in seq)}


runs = ["v6c_track_majority", "v6c_win15", "v6c_win15_split"]
table = []
for run in runs:
    for stem in EVAL_CLIPS:
        p = os.path.join(EVAL_DIR, "runs", run, f"{stem}_tracking_final.json")
        if os.path.exists(p):
            table.append({"run": run, "clip": stem[-6:-2], **scorer_score(p, gt[stem])})
T = pd.DataFrame(table)
display(T.drop(columns=["ID 순서"]))
for _, r in T[T.run == "v6c_win15_split"].iterrows():
    print(f"[{r['clip']}] {r['ID 순서']}")
print("\n이상적인 결과: 잡은 비율 1.0 · ID 개수 1 · ID 바뀜 0 (득점자를 처음부터 끝까지 한 ID로 따라감)")

In [ ]:
# ⑦ 0144 인원 급증(1300프레임 근처)이 ROI 때문인지 확인
STEM = "DJI_20250913133612_0144_D"
SHOW = [1270, 1295, 1310, 1320, 1330]
cand44, last44 = roi_candidates(STEM)                       # 약 1~2분
seq44, _ = roi_sequence(cand44, min(last44, CACHE[STEM]["end"]), False, ROI_1A[STEM])
rows44 = track_clip(CACHE[STEM], lambda f: seq44[f], 1, min(last44, CACHE[STEM]["end"]))
lab44 = label_window(rows44, WIN_MAIN)
imgs = read_frames_sequential(video_path_of(STEM), SHOW)
for fid in SHOW:
    upd = max(k for k in cand44 if k <= fid)
    cnt = Counter(TEAM[lab44[(t, fid)]] for t, b, s, c in rows44[fid])
    im = draw_ids(imgs[fid], rows44[fid], lab44, fid)
    cv2.polylines(im, [np.asarray(seq44[fid], np.int32).reshape(-1, 1, 2)], True, (0, 255, 0), 3)
    plt.figure(figsize=(18, 10.2)); plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(f"0144 · f{fid} · ROI(초록)는 f{upd}에 계산한 것 · 인원 {dict(cnt)}", fontsize=14)
    plt.tight_layout(); plt.show()
print("확인할 것: 국민대가 많아지는 프레임에서 초록 ROI가 경기장 밖(사이드라인)까지 덮고 있는지, 덮고 있다면 몇 프레임에 계산한 ROI인지")